# Phase 6 (Abstractive Summarization) — Run #2: Phase-1 **Encoder-Decoder model 01** (bidirectional encoder, 6 + 6 layers) with **Houlsby adapters**

Config-driven like the other notebooks: edit **Cell 2** only. Same data pipeline, decoding sweep, evaluation and submission as run #1 (Dec-02 + LoRMA), so the two are directly comparable. Only the backbone and the adaptation method change. This run is the second **KD-teacher candidate** (`EXPORT_TEACHER=True`).

### Why this is the natural model for the task
The Phase-1 encoder-decoder was pretrained exactly in the shape we need: a **768-token document window into the bidirectional encoder**, then the decoder generates the **next 256 tokens** from `[BOS]` while cross-attending to the encoder. Summarisation keeps that interface and only changes what the decoder must write: the part of the summary that came from the chunk instead of the text that follows it.

| Audit finding | Consequence here |
|---|---|
| Summaries far exceed any context (only 1 % of pairs fit) | Same **chunk-level** pairs as run #1: 512-token chunk → the summary sentences aligned to it, concatenated in document order at test time |
| Target median 114 tokens, p90 277 (512-token chunks) | `TGT_MAX = 255`: the decoder's trained length is 256 (`[BOS]` + 255), so no RoPE extrapolation in the decoder; 12.7 % of targets are cut at a sentence boundary |
| Encoder trained on 768 tokens | Encoder input `[BOS] [part i/n] <512 chunk tokens>` ≈ 520 tokens, within the trained length. The chunking is identical to run #1, so the KD keys (`uid|chunk`) match for any student |
| Extractive references, BERTScore sees ~410 words, ROUGE-1 peaks at ~1.25× length | Same decoding controls and sweep (incl. the per-chunk minimum length `min_new_all`), same length budget, same document order |

### Houlsby adapters (Module 4.1 H; Houlsby et al. 2019)
Every sub-layer output projection gets a serial bottleneck adapter **before the residual add**: `out = y + W_up·GELU(W_down·y)`, `y = W₀x` frozen, with `W_up = 0` at init, so step 0 is exactly the Phase-1 model (checked). Sites:
* encoder blocks: after self-attention (`o_proj`) and after the FFN (`down_proj`);
* decoder blocks: after self-attention, after **cross-attention** (both `o_proj`) and after the FFN.

That is 6×2 + 6×3 = **30 adapters**, bottleneck 64 → ~2.0M trainable weights (+ the RMSNorm gains, as in Houlsby et al.). This is about the same budget as run #1's LoRMA (1.97M), so the comparison is fair. `sites=("ffn",)` gives the Pfeiffer configuration (FFN adapters only). Embeddings / LM head (tied) are frozen.

### Training and generation
* Encoder: bidirectional with the key-padding mask (right padding). Decoder: `[BOS] + target`, labels `target + [EOS]`, causal, cross-attention masked to the real encoder tokens. Loss = mean NLL over target tokens (logits only at labelled positions).
* Generation: the encoder runs **once per chunk**; the cross-attention K/V are precomputed once; the decoder runs with a self-attention KV cache. All rows start at position 0 from `[BOS]`, so no left padding is needed. Cell 19 checks batched + cached generation against one-sequence, no-cache greedy decoding (prompts of different lengths, i.e. with encoder padding).

### Run it
`SMOKE_TEST=True` first (~10 min), then the full run. Paste the **"PASTE TO CLAUDE"** block.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle, copy, hashlib, bisect, subprocess
from pathlib import Path
from dataclasses import dataclass, asdict, field, fields
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p6_encdec_adapter"
METHOD     = "adapter"        # Houlsby adapters (sites attn + ffn); sites=("ffn",) -> Pfeiffer adapters
SMOKE_TEST = False            # True: ~150 steps on a small subset, a few VAL documents generated, NO submission / export

# ---- Phase-1 checkpoint (written by caps_enc_dec_architecture_01.ipynb; must have encoder_causal=False) ----------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data / split (audit) ------------------------------------------------------------------------
HF_REPO        = "Exploration-Lab/CS-781-Capstone"     # Phase_6_Summ/{Train/train.json, Test/test.json}
LOCAL_DATA_DIR = None
P6             = "Phase_6_Summ"
VAL_N, VAL_SALT, VAL_CHECKSUM_EXPECTED = 200, "p6-val-v1", "5720e140c147"
MIN_SUMM_WORDS, MAX_SUMM_WORDS = 20, 5000       # training pairs only (VAL is kept as in the audit)
MAX_FULL_CHARS = 150_000                        # longer documents (< 1 %) -> first 2/3 + last 1/3 of this many characters

# ---- Chunk pairs ---------------------------------------------------------------------------------
CHUNK_TOKENS = 512           # document tokens per chunk
TGT_MAX      = 255           # [BOS] + 255 = the decoder's trained length 256 (audit: target p90 277; longer ones cut at a sentence boundary)
MIN_ALIGN_F1 = 0.2           # weaker alignments are not assigned to a chunk (audit: 1.9 % of summary sentences)
EMPTY_KEEP   = 0.5           # share of chunks WITHOUT aligned summary text kept as EOS-only examples (resampled every epoch)
TAG_TEMPLATE = "[part {i}/{n}] "
SEP_TEXT     = ""            # not used: the encoder reads the chunk, the decoder writes the summary
N_PROC       = 4

# ---- Training ------------------------------------------------------------------------------------
EPOCHS        = 5            # 6+6 layers, decoder sequences ~150 tokens: cheaper per step than run #1 (check the >> speed line)
MICRO_TOKENS  = 12288        # padded tokens per micro-batch (n_seqs x longest); lower it on OOM
STEP_SEQS     = 32           # sequences per optimiser step (gradient accumulation)
WARMUP_FRAC   = 0.03
GRAD_CLIP     = 1.0
GRAD_CKPT     = False
TIME_BUDGET_MIN = 400        # training stops after this (best epoch kept); leaves ~3 h for evaluation, test and export
HP = {"adapter": dict(lr=1e-3, wd=0.0, bottleneck=64, sites=("attn", "ffn"), stacks=("encoder", "decoder"), train_norms=True)}
SELECT_BY          = "loss"  # "loss" (VAL teacher-forced loss) | "rouge" (quick ROUGE on VAL_GEN_DOCS_EPOCH docs)
VAL_GEN_DOCS_EPOCH = 40

# ---- Generation ----------------------------------------------------------------------------------
GEN = dict(max_chunks=16,        # audit: 16 x 512 reads 95.7 % of all tokens
           max_new=TGT_MAX, min_new_first=32,   # the first chunk must say something (no empty summaries)
           min_new_all=0,                       # [EOS] banned for this many tokens in EVERY chunk (length control; chosen by the sweep)
           no_repeat_ngram=4, rep_penalty=1.1, batch=64,
           len_factor=1.25, dedup=True)
DECODE_SWEEP = [dict(no_repeat_ngram=4, rep_penalty=1.1, min_new_all=0),  dict(no_repeat_ngram=4, rep_penalty=1.1, min_new_all=32),
                dict(no_repeat_ngram=4, rep_penalty=1.1, min_new_all=64), dict(no_repeat_ngram=3, rep_penalty=1.0, min_new_all=32),
                dict(no_repeat_ngram=0, rep_penalty=1.2, min_new_all=32)]
SWEEP_DOCS   = 60            # VAL docs used to pick the decoding config (by (R1 + RL) / 2); [] / 0 = skip
LEN_MODEL    = dict(a=4.538474, b=0.6144775, lo=232.0, hi=2002.1)       # audit power law (words)
RUN_BERTSCORE, BERTSCORE_MODEL = True, "roberta-large"                   # metric only, never used for training
BASELINES = dict(tfidf=dict(r1=0.5749, r2=0.3186, rL=0.3008, bs=0.8533), lead=dict(r1=0.5222, r2=0.2508, rL=0.2437, bs=0.8321),
                 oracle=dict(r1=0.786, r2=0.6042, rL=0.5341, bs=0.8855))

# ---- KD teacher export (run #3) -------------------------------------------------------------------
EXPORT_TEACHER    = True
TEACHER_TOPK      = 8        # top-k logits per gold target token (+ logsumexp), all TR pairs
TEACHER_GEN_PAIRS = 15000    # teacher-generated summaries for this many TR chunks (sequence-level KD)
TEACHER_GEN_MIN   = 90       # time cap (minutes) for the generation part

# ---- Smoke test ----------------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_TRAIN_DOCS, SMOKE_VAL_DOCS = 150, 400, 12

# ---- derived -------------------------------------------------------------------------------------
assert METHOD in HP and SELECT_BY in ("loss", "rouge")
BODY = CHUNK_TOKENS
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} {HP[METHOD]} | chunk {CHUNK_TOKENS} -> target <= {TGT_MAX} | empty keep {EMPTY_KEEP} | "
      f"epochs {EPOCHS} | micro {MICRO_TOKENS} tok, step {STEP_SEQS} seqs | smoke={SMOKE_TEST}")

RUN p6_encdec_adapter | METHOD=adapter {'lr': 0.001, 'wd': 0.0, 'bottleneck': 64, 'sites': ('attn', 'ffn'), 'stacks': ('encoder', 'decoder'), 'train_norms': True} | chunk 512 -> target <= 255 | empty keep 0.5 | epochs 5 | micro 12288 tok, step 32 seqs | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- as in Phases 3-4.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_enc_dec_architecture_01.ipynb` (Cell 3)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# ===================== Cell 3: model =====================
@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_encoder_layers: int = 6
    n_decoder_layers: int = 6
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8
    head_dim: int = 64
    ff_mult: float = 8 / 3
    d_ff: Optional[int] = None
    max_seq_len: int = 1024          # RoPE table size; must cover encoder_len and decoder_len
    encoder_len: int = 768
    decoder_len: int = 256
    norm_type: str = "rmsnorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"           # rope | learned
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"         # swiglu | gelu
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True
    encoder_causal: bool = False     # False = bidirectional encoder
    mlm_head: bool = True            # BERT-style MLM head on the encoder (auxiliary objective)
    arch_name: str = "encdec_bidir_mlm_v2"

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim
        assert self.n_head % self.n_kv_head == 0
        assert self.max_seq_len >= max(self.encoder_len, self.decoder_len)
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return ((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype) * self.g + self.b

def build_norm(cfg):
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "layernorm":
        return LayerNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)


class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, theta=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids=None):
        T = q.shape[2]
        if position_ids is None:
            cos = self.cos[:T][None, None]; sin = self.sin[:T][None, None]
        else:                                    # position_ids: [B, T] or [1, T]
            cos = self.cos[position_ids].unsqueeze(1); sin = self.sin[position_ids].unsqueeze(1)
        dt = q.dtype
        q = self._rotate(q.float(), cos, sin); k = self._rotate(k.float(), cos, sin)
        return q.to(dt), k.to(dt)


class Attention(nn.Module):
    # Self-attention (optionally causal, with KV cache) or cross-attention (context / precomputed static_kv).
    def __init__(self, cfg, causal):
        super().__init__()
        self.cfg, self.causal = cfg, causal
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, nh):
        B, T, _ = x.shape
        return x.view(B, T, nh, self.head_dim).transpose(1, 2)

    def project_kv(self, src):
        return self._split(self.k_proj(src), self.n_kv_head), self._split(self.v_proj(src), self.n_kv_head)

    def _attend(self, q, k, v, causal, key_mask):
        T, S = q.size(2), k.size(2)
        dp = self.cfg.attn_dropout if self.training else 0.0
        mask, sdpa_causal = None, False
        if causal and T > 1:
            if T == S and key_mask is None:
                sdpa_causal = True
            else:   # general (offset) causal mask: query i sees keys <= i + (S - T)
                mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril(diagonal=S - T).view(1, 1, T, S)
        if key_mask is not None:                 # key_mask: [B, S], True/1 = real token
            km = key_mask.to(torch.bool).view(key_mask.size(0), 1, 1, S)
            mask = km if mask is None else (mask & km)
        if self.cfg.use_sdpa:
            return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp, is_causal=sdpa_causal)
        scores = (q.float() @ k.float().transpose(-2, -1)) * self.scale
        if sdpa_causal:
            mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril().view(1, 1, T, S)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        p = F.dropout(F.softmax(scores, dim=-1), dp, self.training).to(q.dtype)
        return p @ v

    def forward(self, x, rope=None, position_ids=None, key_mask=None, kv_cache=None,
                context=None, static_kv=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        is_cross = (context is not None) or (static_kv is not None)
        if static_kv is not None:
            k, v = static_kv
        elif context is not None:
            k, v = self.project_kv(context)
        else:
            k, v = self.project_kv(x)
            if rope is not None:
                q, k = rope(q, k, position_ids=position_ids)
            if kv_cache is not None:
                k = torch.cat([kv_cache[0], k], dim=2)
                v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = None if is_cross else (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1); v = v.repeat_interleave(rep, dim=1)
        out = self._attend(q, k, v, causal=(self.causal and not is_cross), key_mask=key_mask)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GELUMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))

def build_mlp(cfg):
    return SwiGLU(cfg) if cfg.mlp_type == "swiglu" else GELUMLP(cfg)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=cfg.encoder_causal)
        self.norm2, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, rope=None, key_mask=None, position_ids=None):
        h, _ = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, key_mask=key_mask)
        x = x + h
        return x + self.mlp(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=True)
        self.norm2, self.cross_attn = build_norm(cfg), Attention(cfg, causal=False)
        self.norm3, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, enc_out=None, cross_kv=None, enc_mask=None, rope=None,
                position_ids=None, kv_cache=None):
        h, new_self = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, kv_cache=kv_cache)
        x = x + h
        if cross_kv is not None:
            h, _ = self.cross_attn(self.norm2(x), static_kv=cross_kv, key_mask=enc_mask)
        else:
            h, _ = self.cross_attn(self.norm2(x), context=enc_out, key_mask=enc_mask)
        x = x + h
        return x + self.mlp(self.norm3(x)), new_self


class Encoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_encoder_layers)])
        self.norm_f = build_norm(cfg)
    def forward(self, input_ids, attention_mask=None, position_ids=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        for blk in self.blocks:
            x = blk(x, rope=self.rope, key_mask=attention_mask, position_ids=position_ids)
        return self.norm_f(x)

class Decoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_decoder_layers)])
        self.norm_f = build_norm(cfg)

    def precompute_cross_kv(self, enc_out):
        # Cross-attention K/V depend only on the encoder output: compute once per encoder pass.
        return [blk.cross_attn.project_kv(enc_out) for blk in self.blocks]

    def forward(self, input_ids, enc_out=None, enc_mask=None, position_ids=None,
                kv_caches=None, cross_kvs=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, enc_out=enc_out,
                        cross_kv=cross_kvs[i] if cross_kvs is not None else None,
                        enc_mask=enc_mask, rope=self.rope, position_ids=position_ids,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class Seq2SeqLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = Encoder(cfg, self.tok_emb)
        self.decoder = Decoder(cfg, self.tok_emb)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        # BERT-style transform before the (tied) output projection for MLM.
        self.mlm_transform = (nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model, bias=True), nn.GELU(),
                                            build_norm(cfg)) if cfg.mlm_head else None)

    def encode(self, enc_ids, enc_mask=None):
        return self.encoder(enc_ids, attention_mask=enc_mask)

    def forward(self, enc_ids, dec_ids, enc_mask=None):
        enc_out = self.encode(enc_ids, enc_mask)
        h, _ = self.decoder(dec_ids, enc_out=enc_out, enc_mask=enc_mask)
        return self.lm_head(h)

    def mlm_logits_at(self, enc_hidden, select):
        # Project only the masked positions -> [N, V] (avoids a full [B, T, V] tensor).
        return self.lm_head(self.mlm_transform(enc_hidden[select]))


def init_weights(model, cfg):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    # GPT-2-style residual scaling, per stack: encoder blocks add 2 residual branches, decoder blocks add 3.
    s_enc = 1.0 / math.sqrt(2 * cfg.n_encoder_layers)
    s_dec = 1.0 / math.sqrt(3 * cfg.n_decoder_layers)
    with torch.no_grad():
        for blk in model.encoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_enc); blk.mlp.down_proj.weight.mul_(s_enc)
        for blk in model.decoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_dec); blk.cross_attn.o_proj.weight.mul_(s_dec)
            blk.mlp.down_proj.weight.mul_(s_dec)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())   # .parameters() de-duplicates tied weights
    emb = model.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


# ---- Checkpoint helpers (with the causal/bidirectional guard) ----
def cfg_from_dict(d):
    d = dict(d)
    if "encoder_causal" not in d:
        raise ValueError("Checkpoint config has no 'encoder_causal' key. This is probably an old causal-encoder "
                         "checkpoint. Refusing to guess: the flag is a mask, not a weight, so a wrong guess would "
                         "load silently and run the encoder in a mode it was never trained in. Add "
                         "ckpt['config']['encoder_causal'] = True/False manually if you know which run it is.")
    d.setdefault("mlm_head", False)          # pre-v2 checkpoints have no MLM head
    d.setdefault("arch_name", "legacy")
    known = {f.name for f in fields(ModelConfig)}
    dropped = sorted(set(d) - known)
    if dropped:
        print("cfg_from_dict: ignoring unknown keys", dropped)
    return ModelConfig(**{k: v for k, v in d.items() if k in known})

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_model_from_ckpt(path, expect_encoder_causal=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = cfg_from_dict(ck["config"])
    if expect_encoder_causal is not None and cfg.encoder_causal != expect_encoder_causal:
        raise ValueError(f"{path}: encoder_causal={cfg.encoder_causal}, expected {expect_encoder_causal}")
    model = Seq2SeqLM(cfg)
    model.load_state_dict(ck["model"], strict=True)
    st = ck.get("state", {})
    print(f"Loaded {path} | arch={cfg.arch_name} | encoder_causal={cfg.encoder_causal} | "
          f"enc_len={cfg.encoder_len} dec_len={cfg.decoder_len} | step={st.get('step', ck.get('step'))} | "
          f"val_loss={st.get('val_loss', ck.get('val_loss'))}")
    return model.to(device).eval(), cfg, ck

_c = ModelConfig(vocab_size=VOCAB_SIZE)
_pc = count_params(Seq2SeqLM(_c))
print(f"Default config params: total={_pc['total']/1e6:.2f}M (embedding {_pc['embedding']/1e6:.2f}M)")
del _c

Default config params: total=61.49M (embedding 16.38M)


## Load the Phase-1 checkpoint

In [6]:
# Cell 12 — Load the Phase-1 encoder-decoder checkpoint and sanity-check BOTH stacks
lm, lm_cfg, _ck = load_model_from_ckpt(CKPT_PATH, expect_encoder_causal=False)       # refuses a causal-encoder checkpoint
del _ck
pc = count_params(lm)
print(f"Phase-1 model | arch={lm_cfg.arch_name} | enc layers={lm_cfg.n_encoder_layers} dec layers={lm_cfg.n_decoder_layers} "
      f"d_model={lm_cfg.d_model} heads {lm_cfg.n_head}:{lm_cfg.n_kv_head} pos={lm_cfg.pos_type} | enc_len {lm_cfg.encoder_len} "
      f"dec_len {lm_cfg.decoder_len} max_seq_len {lm_cfg.max_seq_len} | total {pc['total']/1e6:.2f}M")
# Seq2seq sanity: the decoder continues an encoded legal sentence. Untrained / mis-loaded weights sit near ln(V).
_sent = ("The appellant was convicted by the Sessions Court under Section 302 of the Indian Penal Code and the conviction was "
         "confirmed by the High Court. The appeal is allowed and the impugned judgment of the High Court is set aside.")
_ids = tokenizer.encode(_sent); _h = len(_ids) // 2
_enc = torch.tensor([_ids[:_h]], device=device); _tgt = _ids[_h:]
_din = torch.tensor([[BOS_ID] + _tgt[:-1]], device=device); _y = torch.tensor([_tgt], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _lg = lm(_enc, _din)
_l = F.cross_entropy(_lg.float().view(-1, _lg.size(-1)), _y.view(-1)).item()
print(f"seq2seq continuation loss on a legal sentence: {_l:.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l < 9.0, "checkpoint looks untrained / mis-loaded"
assert VOCAB_SIZE < 65536, "token ids are stored as uint16"
assert TGT_MAX + 1 <= lm_cfg.decoder_len, f"TGT_MAX {TGT_MAX} + [BOS] exceeds the decoder's trained length {lm_cfg.decoder_len}"
_SPEC = set(getattr(tokenizer, "special_to_id", {}).values())
BAN_IDS = sorted((_SPEC | {PAD_ID, BOS_ID, UNK_ID}) - {EOS_ID})              # never generated
print("tokens never generated:", BAN_IDS)

Loaded /kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt | arch=encdec_bidir_mlm_v2 | encoder_causal=False | enc_len=768 dec_len=256 | step=15488 | val_loss=1.9592824522405863
Phase-1 model | arch=encdec_bidir_mlm_v2 | enc layers=6 dec layers=6 d_model=512 heads 8:8 pos=rope | enc_len 768 dec_len 256 max_seq_len 1024 | total 61.49M
seq2seq continuation loss on a legal sentence: 1.793  (ln V = 10.37)
tokens never generated: [0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122]


## Phase-6 data, the fixed validation split, cleaning
`split_train` and `_h` are copied **verbatim** from the audit (Cell 5); the checksum assert guarantees that every Phase-6 run is scored on the same 200 documents. Cleaning (too-short / too-long summaries, summaries longer than their document) is applied to the **training** part only.

In [7]:
# Cell 13 — Load train / test, split, clean
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

def fetch(rel):
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet, or download {P6} and set LOCAL_DATA_DIR.") from ex

ID_KEYS  = ("id", "doc_id", "document_id", "case_id")
DOC_KEYS = ("document", "text", "judgment", "judgement", "source", "case_text", "content")
SUM_KEYS = ("summary", "reference_summary", "abstract", "target", "headnote")
def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default
def _as_text(v): return "\n".join(str(t) for t in v) if isinstance(v, list) else v

def _read_json_any(path):
    txt = Path(path).read_text(encoding="utf-8")
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]

def parse_docs(raw, split, has_sum):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): items = raw["data"]
        else:
            items = []
            for k, v in raw.items():
                v = dict(v) if isinstance(v, dict) else {"document": v}
                v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        rid = _first(e, ID_KEYS, i)
        doc = _as_text(_first(e, DOC_KEYS))
        if not isinstance(doc, str): raise ValueError(f"{split} item {rid}: no document field. Keys: {list(e)}")
        summ = _as_text(_first(e, SUM_KEYS)) if has_sum else None
        if has_sum and not isinstance(summ, str): raise ValueError(f"{split} item {rid}: no summary field. Keys: {list(e)}")
        docs.append(dict(uid=f"{split}::{i}", id=rid, doc=doc, summ=summ))
    return docs

def load_split(rel, split, has_sum):
    raw = _read_json_any(fetch(rel))
    print(f"  {rel}: {type(raw).__name__} of {len(raw)}")
    return parse_docs(raw, split, has_sum)

TRAIN = load_split(f"{P6}/Train/train.json", "train", True)
TEST_ALL = load_split(f"{P6}/Test/test.json", "test", False)

# ---- verbatim from the audit (Cell 5) ----
def _norm(t): return re.sub(r"\s+", " ", t).strip().lower()
def _h(t):    return hashlib.md5(_norm(t).encode("utf-8")).hexdigest()
def split_train(train, val_n=VAL_N, salt=VAL_SALT):
    # Whole duplicate-document groups go to one side; group order = md5(salt :: normalised-text hash).
    groups = defaultdict(list)
    for d in train: groups[_h(d["doc"])].append(d)
    order = sorted(groups, key=lambda g: hashlib.md5(f"{salt}::{g}".encode()).hexdigest())
    val, val_keys = [], set()
    for g in order:
        if len(val) >= val_n: break
        val.extend(groups[g]); val_keys.add(g)
    tr = [d for d in train if _h(d["doc"]) not in val_keys]
    checksum = hashlib.md5("|".join(sorted(str(d["id"]) for d in val)).encode()).hexdigest()[:12]
    return tr, val, checksum
# ------------------------------------------

TR_ALL, VAL_ALL, _cs = split_train(TRAIN)
print(f"split: TR {len(TR_ALL)} | VAL {len(VAL_ALL)} | checksum {_cs}")
assert _cs == VAL_CHECKSUM_EXPECTED, f"VAL split differs from the audit ({_cs} != {VAL_CHECKSUM_EXPECTED})"

def _clean(docs):
    keep, drop = [], Counter()
    seen = set()
    for d in docs:
        sw, dw = len(d["summ"].split()), len(d["doc"].split())
        key = (_h(d["doc"]), _h(d["summ"]))
        if sw < MIN_SUMM_WORDS: drop["short summary"] += 1; continue
        if sw > MAX_SUMM_WORDS: drop["very long summary"] += 1; continue
        if sw > dw: drop["summary longer than document"] += 1; continue
        if key in seen: drop["exact duplicate pair"] += 1; continue
        seen.add(key); keep.append(d)
    return keep, dict(drop)
TR_ALL, _dropped = _clean(TR_ALL)
print(f"training docs after cleaning: {len(TR_ALL)} | dropped {_dropped}")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

  Phase_6_Summ/Train/train.json: list of 7030


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

  Phase_6_Summ/Test/test.json: list of 100
split: TR 6830 | VAL 200 | checksum 5720e140c147
training docs after cleaning: 6759 | dropped {'exact duplicate pair': 25, 'very long summary': 43, 'summary longer than document': 1, 'short summary': 2}


## Sentence alignment → chunk-level training pairs
`split_sentences`, `content` and `align_doc` are the audit's (Cell 8), unchanged. For every training document: tokenise it **with character offsets**, cut it into 512-token chunks, align each summary sentence to its best document sentence, and give the sentence to the chunk containing that source sentence. A chunk's target is its sentences **in summary order**, cut at a sentence boundary to `TGT_MAX` tokens (a single over-long sentence is cut and gets no `[EOS]`, so the model never learns to stop mid-sentence). Tokenisation and alignment run in `N_PROC` processes and are cached.

In [8]:
# Cell 14 — Sentence splitter + aligner (verbatim from the audit, Cell 8)
_WRE = re.compile(r"[a-z0-9]+")
def words(t): return _WRE.findall(t.lower())
_ABBR = (r"mr|mrs|ms|dr|hon'ble|honble|no|nos|sr|jr|ltd|pvt|co|corp|inc|vs|v|viz|i\.e|e\.g|etc|cf|art|arts|sec|secs|s|ss|cl|"
         r"para|paras|p|pp|vol|ors|anr|govt|st|rs|prof|smt|shri|sri|ch|app|crl|cr|civ|j|jj|cj|mad|bom|cal|del|all|scc|scr|air|"
         r"u/s|r/w|w\.e\.f|addl|asstt|dist|distt|dept|reg|regd|ft|sq|nos|approx|ibid|supra|id|cri|lj|cri\.l\.j|misc|wp|slp")
_ABBR_END = re.compile(r"(?:\b(?:" + _ABBR + r")|\b[A-Za-z])\.$|^\s*\(?\d{1,3}[.)]$", re.I)
_SPLIT = re.compile(r"(?<=[.?!])[\"')\]]*\s+(?=[\"'(\[]?[A-Z0-9])|\n\s*\n|\n(?=\s*(?:\d{1,3}[.)]\s|\([a-z0-9]{1,4}\)\s))")

def split_sentences(text):
    # Returns [(char_start, sentence)]. Pieces ending in an abbreviation / initial / short number are merged forward.
    out, start = [], 0
    pieces = []
    for m in _SPLIT.finditer(text):
        pieces.append((start, text[start:m.start()])); start = m.end()
    pieces.append((start, text[start:]))
    buf_s, buf = None, ""
    for s, p in pieces:
        if buf_s is None: buf_s, buf = s, p
        else: buf = buf + " " + p
        if not _ABBR_END.search(buf.rstrip()):
            if buf.strip(): out.append((buf_s, buf.strip()))
            buf_s, buf = None, ""
    if buf_s is not None and buf.strip(): out.append((buf_s, buf.strip()))
    return out

_STOP = set(("the a an of to in and or for on by with at from as is was were be been are that this these those it its which who "
             "whom whose has have had not no but if than then so such any all also there their they he she his her him we our us you "
             "your i me my shall may can could would should will said upon into under over between after before against within "
             "without about being do does did same other further thereof therein herein hereby whereas").split())
def content(t): return {w for w in _WRE.findall(t.lower()) if w not in _STOP and len(w) > 1}

def align_doc(doc, summ):
    ds, ss = split_sentences(doc), split_sentences(summ)
    dsets = [content(t) for _, t in ds]
    inv = defaultdict(list)
    for j, st in enumerate(dsets):
        for w in st: inv[w].append(j)
    res = []
    for k, (_, t) in enumerate(ss):
        q = content(t)
        if len(q) < 2: continue
        hits = Counter()
        for w in q:
            for j in inv.get(w, ()): hits[j] += 1
        best_j, best_f = -1, 0.0
        for j, c in hits.items():
            f = 2 * c / (len(q) + len(dsets[j]))
            if f > best_f: best_j, best_f = j, f
        if best_j >= 0:
            res.append(dict(k=k, j=best_j, f=best_f, pos=ds[best_j][0] / max(len(doc), 1), char=ds[best_j][0], words=len(t.split())))
    return ds, ss, res

In [9]:
# Cell 15 — Tokenise documents into chunks (+ chunk targets for documents with a summary); cache; prompt pieces
def cut_text(t):
    if len(t) <= MAX_FULL_CHARS: return t
    h = 2 * MAX_FULL_CHARS // 3
    return t[:h] + "\n" + t[-(MAX_FULL_CHARS - h):]

def _fit_target(sents):
    # sentences (summary order) -> (token ids <= TGT_MAX, truncated?) ; whole sentences while they fit
    chosen, ids = [], []
    for s in sents:
        cand = tokenizer.encode(" ".join(chosen + [s]))
        if len(cand) <= TGT_MAX: chosen.append(s); ids = cand
        else: break
    if chosen: return ids, False
    return tokenizer.encode(sents[0])[:TGT_MAX], True

_DOCS_W, _WITH_TGT = None, False
def _work(i):
    d = _DOCS_W[i]
    text = cut_text(d["doc"])
    ids, offs = tokenizer.encode(text, return_offsets=True)
    ids = list(ids)
    if not ids: ids, offs = [EOS_ID], [(0, 0)]
    starts = list(range(0, len(ids), BODY))
    cchar = [offs[s][0] for s in starts]
    chunks = [np.asarray(ids[s:s + BODY], dtype=np.uint16) for s in starts]
    tgts, trunc = None, None
    if _WITH_TGT and d["summ"] is not None:
        _, ss, res = align_doc(text, d["summ"])
        per = defaultdict(set)
        for r in res:
            if r["f"] >= MIN_ALIGN_F1:
                per[max(0, bisect.bisect_right(cchar, r["char"]) - 1)].add(r["k"])
        tgts, trunc = [], []
        for j in range(len(starts)):
            if not per.get(j): tgts.append(None); trunc.append(False); continue
            t_ids, tr = _fit_target([ss[k][1] for k in sorted(per[j])])
            tgts.append(np.asarray(t_ids, dtype=np.uint16)); trunc.append(tr)
    return dict(chunks=chunks, tgts=tgts, trunc=trunc, ntok=len(ids), words=len(d["doc"].split()))

def prepare_docs(docs, cache_name, with_targets):
    global _DOCS_W, _WITH_TGT
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"p6_{cache_name}_v{VOCAB_SIZE}_c{CHUNK_TOKENS}_t{TGT_MAX}_a{MIN_ALIGN_F1}_m{MAX_FULL_CHARS}_{int(with_targets)}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    key = lambda d: (d["uid"], len(d["doc"]), len(d["summ"] or ""))
    todo = [i for i, d in enumerate(docs) if cache.get(d["uid"], (None,))[0] != key(d)]
    if todo:
        _DOCS_W, _WITH_TGT = docs, with_targets
        order = sorted(todo, key=lambda i: -len(docs[i]["doc"]))
        res = None
        if N_PROC > 1:
            try:
                with mp.get_context("fork").Pool(N_PROC) as pool:
                    res = list(tqdm(pool.imap(_work, order, chunksize=8), total=len(order), desc=f"prepare {cache_name}", leave=False))
            except Exception as ex:
                print("parallel preparation failed (", type(ex).__name__, ex, ") -> serial"); res = None
        if res is None:
            res = [_work(i) for i in tqdm(order, desc=f"prepare {cache_name}", leave=False)]
        for i, r in zip(order, res): cache[docs[i]["uid"]] = (key(docs[i]), r)
        pickle.dump(cache, open(cp, "wb"))
    for d in docs: d.update(cache[d["uid"]][1])
    return docs

_TAG_CACHE = {}
def tag_ids(j, n):
    k = (j, n)
    if k not in _TAG_CACHE: _TAG_CACHE[k] = tokenizer.encode(TAG_TEMPLATE.format(i=j + 1, n=n))
    return _TAG_CACHE[k]
def prompt_ids(d, j):
    # ENCODER input of chunk j: [BOS] [part i/n] <chunk tokens>
    return [BOS_ID] + tag_ids(j, len(d["chunks"])) + d["chunks"][j].astype(np.int64).tolist()
MAX_PROMPT = 1 + len(tag_ids(998, 999)) + BODY
assert MAX_PROMPT <= lm_cfg.max_seq_len, "encoder input exceeds the RoPE table"
if MAX_PROMPT > lm_cfg.encoder_len: print(f"WARNING: encoder input {MAX_PROMPT} > trained encoder length {lm_cfg.encoder_len}")
print(f"encoder input <= {MAX_PROMPT} tokens (trained {lm_cfg.encoder_len}) | decoder [BOS] + target <= {TGT_MAX + 1} (trained {lm_cfg.decoder_len})")

def pair_stats(docs, title):
    n_ch = np.asarray([len(d["chunks"]) for d in docs])
    t = [len(x) for d in docs for x in d["tgts"] if x is not None]
    emp = np.mean([x is None for d in docs for x in d["tgts"]])
    tr = np.mean([f for d in docs for x, f in zip(d["tgts"], d["trunc"]) if x is not None])
    print(f"{title}: {len(docs)} docs | chunks/doc mean {n_ch.mean():.1f} median {np.median(n_ch):.0f} | chunk pairs with a target {len(t)} "
          f"| empty chunks {100 * emp:.1f}% | target tokens median {np.median(t):.0f} p90 {np.percentile(t, 90):.0f} | "
          f"single-sentence cut {100 * tr:.1f}%")

encoder input <= 524 tokens (trained 768) | decoder [BOS] + target <= 256 (trained 256)


## Adapters, batches, loss, generation
* **Batches**: encoder inputs right-padded with a key-padding mask; decoder inputs `[BOS] + target[:-1]`, labels `target` (which ends with `[EOS]` unless a single over-long sentence was cut), right-padded with −100 labels. Micro-batches group similar lengths up to `MICRO_TOKENS` (n × (encoder + decoder length)).
* **Generation**: one encoder pass per chunk, cross-attention K/V precomputed once, then greedy decoding with the self-attention KV cache and the same controls as run #1 (no-repeat-n-gram, repetition penalty on this chunk's generated tokens, `[EOS]` banned for the first `min_new_first` tokens of the first chunk / `min_new_all` tokens of every chunk).

In [10]:
# Cell 16 — Houlsby bottleneck adapters (the Phase-5 code, unchanged) + model construction
class AdapterAfter(nn.Module):
    # y = W x (frozen);  out = y + W_up f(W_down y).  W_up = 0 at init  =>  out = y (exactly the Phase-1 sub-layer).
    def __init__(self, base: nn.Linear, bottleneck: int):
        super().__init__()
        d = base.out_features
        self.base = base
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5)); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))

def apply_adapter(module, targets, make):
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n

def build_model():
    # Fresh copy of the Phase-1 encoder-decoder; everything frozen except the adapters (+ norm gains).
    model = copy.deepcopy(lm).cpu()
    for p in model.parameters(): p.requires_grad = False
    hp = HP[METHOD]
    ref = copy.deepcopy(model)
    tg = tuple(t for site, t in (("attn", "o_proj"), ("ffn", "down_proj")) if site in hp["sites"])
    n = 0
    for st in hp["stacks"]:
        n += apply_adapter(getattr(model, st), tg, lambda l: AdapterAfter(l, hp["bottleneck"]))
    n_ad = sum(p.numel() for m_ in model.modules() if isinstance(m_, AdapterAfter) for k, p in m_.named_parameters() if not k.startswith("base."))
    n_norm = 0
    if hp["train_norms"]:
        for st in hp["stacks"]:
            for m_ in getattr(model, st).modules():
                if isinstance(m_, (RMSNorm, LayerNorm)):
                    for p in m_.parameters(): p.requires_grad = True; n_norm += p.numel()
    print(f"adapters after {tg} in {hp['stacks']}: {n} modules, bottleneck {hp['bottleneck']} -> {n_ad:,} adapter params + {n_norm:,} norm gains")
    _e = torch.randint(10, VOCAB_SIZE, (2, 40)); _d = torch.randint(10, VOCAB_SIZE, (2, 12)); _m = torch.ones_like(_e); _m[1, 30:] = 0
    with torch.no_grad():
        a = model.eval()(_e, _d, _m); b = ref.eval()(_e, _d, _m)
    err = (a - b).abs().max().item()
    print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
    assert err < 1e-4, "adapter does not start from the Phase-1 function"
    del ref
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%)")
    model.n_total, model.n_trainable = tot, tr
    return model

In [11]:
# Cell 17 — Training pairs, micro-batches, loss
def make_pairs(docs, seed, empty_keep=EMPTY_KEEP):
    # -> list of (doc index, chunk index). Every chunk with a target + a random share of the empty ones (EOS-only target).
    rng = random.Random(seed); pairs = []
    for di, d in enumerate(docs):
        for j, t in enumerate(d["tgts"]):
            if t is not None or rng.random() < empty_keep: pairs.append((di, j))
    return pairs

def seq_of(docs, pair):
    # -> (encoder ids, decoder labels = target [+ EOS])
    di, j = pair; d = docs[di]
    t = d["tgts"][j]
    tgt = ([] if t is None else t.astype(np.int64).tolist()) + ([] if (t is not None and d["trunc"][j]) else [EOS_ID])
    return prompt_ids(d, j), tgt

def make_microbatches(docs, pairs, shuffle, seed=0, budget=None):
    budget = budget or MICRO_TOKENS
    lens = [len(prompt_ids(docs[di], j)) + (0 if docs[di]["tgts"][j] is None else len(docs[di]["tgts"][j])) + 1 for di, j in pairs]
    idx = list(range(len(pairs))); rng = random.Random(seed)
    if shuffle: rng.shuffle(idx)
    batches, pool = [], 2000
    for p in range(0, len(idx), pool):
        part = sorted(idx[p:p + pool], key=lambda i: lens[i])
        cur, longest = [], 0
        for i in part:
            if cur and (len(cur) + 1) * max(longest, lens[i]) > budget:
                batches.append(cur); cur, longest = [], 0
            cur.append(i); longest = max(longest, lens[i])
        if cur: batches.append(cur)
    if shuffle: rng.shuffle(batches)
    return [[pairs[i] for i in b] for b in batches]

def build_batch(docs, pairs):
    encs, labs = zip(*[seq_of(docs, pr) for pr in pairs])
    E, T = max(len(e) for e in encs), max(len(l) for l in labs)
    enc = np.full((len(encs), E), PAD_ID, dtype=np.int64); em = np.zeros((len(encs), E), dtype=np.int64)
    din = np.full((len(encs), T), PAD_ID, dtype=np.int64); y = np.full((len(encs), T), -100, dtype=np.int64)
    for i, (e, l) in enumerate(zip(encs, labs)):
        enc[i, :len(e)] = e; em[i, :len(e)] = 1
        dl = [BOS_ID] + l[:-1]
        din[i, :len(dl)] = dl; y[i, :len(l)] = l
    t = lambda a: torch.from_numpy(a).to(device)
    return dict(enc=t(enc), enc_mask=t(em), dec=t(din), y=t(y), ntok=int(enc.size + din.size))

def forward_hidden(model, bt):
    # decoder hidden states [N, T, d] and labels; the decoder is causal, right padding needs no self-attention mask
    enc_out = model.encode(bt["enc"], bt["enc_mask"])
    h, _ = model.decoder(bt["dec"], enc_out=enc_out, enc_mask=bt["enc_mask"])
    return h, bt["y"]

def batch_loss(model, bt):
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        h, y = forward_hidden(model, bt)
        sel = y != -100
        logits = model.lm_head(h[sel])
    return F.cross_entropy(logits.float(), y[sel], reduction="sum"), int(sel.sum().item())

@torch.no_grad()
def eval_loss(model, docs, pairs):
    model.eval(); tot, n = 0.0, 0
    for b in make_microbatches(docs, pairs, shuffle=False, budget=2 * MICRO_TOKENS):
        l, k = batch_loss(model, build_batch(docs, b)); tot += l.item(); n += k
    return tot / max(n, 1)

In [12]:
# Cell 18 — Batched greedy generation (encoder once, cached cross K/V + self-attention KV cache), summarisation, ROUGE
@torch.no_grad()
def generate(model, prompts, max_new, min_new=None, no_repeat_ngram=0, rep_penalty=1.0, amp=True):
    # prompts: list of ENCODER token-id lists. Returns generated ids per prompt (without EOS).
    model.eval(); B = len(prompts); L = max(len(p) for p in prompts)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=device); mask = torch.zeros((B, L), dtype=torch.long, device=device)
    for i, p in enumerate(prompts):
        ids[i, :len(p)] = torch.tensor(p, device=device); mask[i, :len(p)] = 1
    min_new_t = torch.tensor(min_new if min_new is not None else [0] * B, device=device)
    ban = torch.tensor(BAN_IDS, device=device, dtype=torch.long)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP and amp):
        enc_out = model.encode(ids, mask)
        cross = model.decoder.precompute_cross_kv(enc_out)
    dec = torch.full((B, 1), BOS_ID, dtype=torch.long, device=device); caches = None
    out = [[] for _ in range(B)]; done = [False] * B; bans = [defaultdict(set) for _ in range(B)]
    seen = None; n = no_repeat_ngram
    for t in range(max_new):
        pos = torch.full((B, 1), t, dtype=torch.long, device=device)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP and amp):
            h, caches = model.decoder(dec, enc_mask=mask, position_ids=pos, kv_caches=caches, cross_kvs=cross)
            lg = model.lm_head(h[:, -1]).float()
        if seen is None: seen = torch.zeros(lg.shape, dtype=torch.bool, device=device)
        if rep_penalty != 1.0:
            lg = torch.where(seen, torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty), lg)
        lg[:, ban] = float("-inf")
        lg[:, EOS_ID] = torch.where(t < min_new_t, torch.full_like(lg[:, EOS_ID], float("-inf")), lg[:, EOS_ID])
        if n > 0 and t >= n - 1:
            rows, cols = [], []
            for b in range(B):
                if done[b]: continue
                for tok in bans[b].get(tuple(out[b][-(n - 1):]) if n > 1 else (), ()): rows.append(b); cols.append(tok)
            if rows: lg[rows, cols] = float("-inf")
        nl = lg.argmax(-1).tolist()
        for b in range(B):
            if done[b]: nl[b] = PAD_ID; continue
            if nl[b] == EOS_ID: done[b] = True; continue
            out[b].append(nl[b])
            if n > 0 and len(out[b]) >= n: bans[b][tuple(out[b][-n:-1]) if n > 1 else ()].add(out[b][-1])
        if all(done): break
        dec = torch.tensor(nl, device=device)[:, None]
        seen[torch.arange(B, device=device), dec[:, 0]] = True
    return out

def select_chunks(n, k):
    # Head + tail + evenly spread middle (the audit's U-shaped source positions).
    if n <= k: return list(range(n))
    h = max(1, k // 4); t = max(1, k // 4); m = k - h - t
    mid = np.linspace(h, n - t - 1, m).round().astype(int).tolist() if m > 0 else []
    return sorted(set(list(range(h)) + mid + list(range(n - t, n))))

def predict_len_words(doc_words):
    return float(np.clip(LEN_MODEL["a"] * max(doc_words, 1) ** LEN_MODEL["b"], LEN_MODEL["lo"], LEN_MODEL["hi"]))

def _dedup(text):
    seen, out = set(), []
    for _, s in split_sentences(text):
        k = " ".join(words(s))
        if k and k in seen: continue
        seen.add(k); out.append(s)
    return " ".join(out)

LAST_GEN_STATS = {}
def summarize_docs(model, docs, gen=None, return_parts=False):
    g = dict(GEN, **(gen or {}))
    jobs = []                                                               # (doc index, chunk index)
    for di, d in enumerate(docs):
        for j in select_chunks(len(d["chunks"]), g["max_chunks"]): jobs.append((di, j))
    prompts = [prompt_ids(docs[di], j) for di, j in jobs]
    first = {di: min(j for dj, j in jobs if dj == di) for di in range(len(docs))}
    order = sorted(range(len(jobs)), key=lambda i: len(prompts[i]))
    outs = [None] * len(jobs); t0 = time.time()
    for s in range(0, len(order), g["batch"]):
        bi = order[s:s + g["batch"]]
        mins = [max(g["min_new_all"], g["min_new_first"] if jobs[i][1] == first[jobs[i][0]] else 0) for i in bi]
        res = generate(model, [prompts[i] for i in bi], g["max_new"], mins, g["no_repeat_ngram"], g["rep_penalty"])
        for i, r in zip(bi, res): outs[i] = r
    parts = defaultdict(list)
    for (di, j), o in zip(jobs, outs): parts[di].append((j, tokenizer.decode(o).strip()))
    LAST_GEN_STATS.update(chunks=len(jobs), empty_share=float(np.mean([len(o) == 0 for o in outs])),
                          tokens_per_chunk=float(np.mean([len(o) for o in outs])), hit_max_share=float(np.mean([len(o) >= g["max_new"] for o in outs])))
    summaries = []
    for di, d in enumerate(docs):
        txt = " ".join(t for _, t in sorted(parts[di]) if t)
        if g["dedup"]: txt = _dedup(txt)
        budget = int(round(g["len_factor"] * predict_len_words(d["words"])))
        w = txt.split()
        summaries.append(" ".join(w[:budget]))
    secs = time.time() - t0
    return (summaries, secs, parts) if return_parts else (summaries, secs)

# ---- ROUGE: fast exact implementation verified against rouge_score (as in the audit, Cell 9) ----
try:
    from rouge_score import rouge_scorer
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score"], check=True)
    from rouge_score import rouge_scorer
_RS = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
def rtok(text): return _RS._tokenizer.tokenize(text)
def lcs_len(a, b):
    if not a or not b: return 0
    masks = {}
    for i, x in enumerate(a): masks[x] = masks.get(x, 0) | (1 << i)
    full = (1 << len(a)) - 1; V = full
    for y in b:
        U = V & masks.get(y, 0)
        V = ((V + U) | (V - U)) & full
    return len(a) - bin(V).count("1")
def _f(o, nc, nr):
    p, r = (o / nc if nc else 0.0), (o / nr if nr else 0.0)
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)
def rouge(cand, ref):
    c, r = rtok(cand), rtok(ref); out = {}
    for n in (1, 2):
        cc, rc = Counter(zip(*[c[i:] for i in range(n)])), Counter(zip(*[r[i:] for i in range(n)]))
        out[f"r{n}"] = _f(sum(min(v, rc[k]) for k, v in cc.items()), sum(cc.values()), sum(rc.values()))
    out["rL"] = _f(lcs_len(r, c), len(c), len(r))
    return out
_c, _r = "the appellant was convicted under section 302 by the court", "the appellant was convicted by the sessions court under section 302"
assert all(abs(_RS.score(_r, _c)[k].fmeasure - rouge(_c, _r)[k2][2]) < 1e-9 for k, k2 in (("rouge1", "r1"), ("rouge2", "r2"), ("rougeL", "rL")))

def rouge_eval(cands, refs):
    R = defaultdict(list)
    for c, r in zip(cands, refs):
        for k, v in rouge(c, r).items(): R[k].append(v)
    return {k: dict(p=float(np.mean([x[0] for x in v])), r=float(np.mean([x[1] for x in v])), f=float(np.mean([x[2] for x in v])))
            for k, v in R.items()}, {k: np.asarray([x[2] for x in v]) for k, v in R.items()}

In [13]:
# Cell 19 — Self-test: batched + cached greedy (encoder padding) == one-sequence, no-cache greedy (on the Phase-1 model)
def _greedy_ref(model, p, steps):
    enc = torch.tensor([p], device=device); dec = [BOS_ID]; out = []
    with torch.no_grad():
        enc_out = model.encode(enc, None)
        for _ in range(steps):
            h, _ = model.decoder(torch.tensor([dec], device=device), enc_out=enc_out)
            lg = model.lm_head(h[:, -1]).float()[0]
            lg[BAN_IDS] = float("-inf")
            t = int(lg.argmax())
            if t == EOS_ID: break
            out.append(t); dec.append(t)
    return out

_tf32 = torch.backends.cuda.matmul.allow_tf32
torch.backends.cuda.matmul.allow_tf32 = False
_tp = [[BOS_ID] + tokenizer.encode("The appellant was convicted by the Sessions Court and"), [BOS_ID] + tokenizer.encode("Held:"),
       [BOS_ID] + tokenizer.encode(VAL_ALL[0]["doc"][:2400])[:420]]
_steps = 24
_b = generate(lm, _tp, _steps, amp=False)
for p, g in zip(_tp, _b):
    r = _greedy_ref(lm, p, _steps)
    assert g == r, f"cached batched generation differs from the reference:\n{g}\n{r}"
torch.backends.cuda.matmul.allow_tf32 = _tf32
print(f"generation self-test OK: batched + cached cross K/V + self-attention cache == reference greedy (encoder inputs of {[len(p) for p in _tp]} tokens)")
print("Phase-1 continuation (no fine-tuning):", repr(tokenizer.decode(_b[0])[:200]))

generation self-test OK: batched + cached cross K/V + self-attention cache == reference greedy (encoder inputs of [10, 4, 421] tokens)
Phase-1 continuation (no fine-tuning): ' 100000000000000000000000'


## Build the data and train

In [14]:
# Cell 20 — Prepare data, build the model, train
seed_everything(SEED)
if SMOKE_TEST:
    _r = random.Random(SEED)
    train_docs = _r.sample(TR_ALL, min(SMOKE_TRAIN_DOCS, len(TR_ALL))); val_docs = VAL_ALL[:SMOKE_VAL_DOCS]
else:
    train_docs, val_docs = TR_ALL, VAL_ALL
_t0 = time.time()
prepare_docs(train_docs, "tr_smoke" if SMOKE_TEST else "tr", True)
prepare_docs(val_docs, "val_smoke" if SMOKE_TEST else "val", True)
print(f"preparation {time.time() - _t0:.0f}s")
pair_stats(train_docs, "TR"); pair_stats(val_docs, "VAL")
val_pairs = make_pairs(val_docs, seed=0, empty_keep=EMPTY_KEEP)

def param_groups(model):
    hp = HP[METHOD]
    ps = [p for p in model.parameters() if p.requires_grad]
    return [dict(params=ps, lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"])]

def quick_rouge(model, docs, n):
    if not n: return None
    s, _ = summarize_docs(model, docs[:n])
    m, _ = rouge_eval(s, [d["summ"] for d in docs[:n]])
    return m

def train_model(model, epochs, max_steps=None):
    opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]
    n_pairs = len(make_pairs(train_docs, seed=SEED + 1))
    steps_per_epoch = math.ceil(n_pairs / STEP_SEQS); total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    print(f"~{n_pairs} training sequences/epoch | ~{steps_per_epoch} steps/epoch ({STEP_SEQS} seqs/step) | total {total} steps")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=None, epoch=0, state=None); t_train = time.time(); tok_seen = 0
    v0 = eval_loss(model, val_docs, val_pairs); print(f"VAL loss before training (Phase-1 model, target tokens): {v0:.4f}")
    for ep in range(1, epochs + 1):
        pairs = make_pairs(train_docs, seed=SEED + ep)
        batches = make_microbatches(train_docs, pairs, shuffle=True, seed=SEED + ep)
        model.train(); t0 = time.time(); ep_sum, ep_tok, acc, acc_tok, acc_loss = 0.0, 0, 0, 0, None
        for bi, b in enumerate(batches):
            bt = build_batch(train_docs, b)
            l, k = batch_loss(model, bt)
            # normalise by the target tokens of the whole step: sum-losses are divided by an estimate (k * STEP_SEQS / len(b))
            scaler.scale(l / max(1.0, k * STEP_SEQS / len(b))).backward()
            ep_sum += l.item(); ep_tok += k; tok_seen += bt["ntok"]; acc += len(b)
            losses.append(l.item() / max(k, 1))
            if acc >= STEP_SEQS or bi == len(batches) - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1; acc = 0
                if step == 20:
                    per = (time.time() - t0) / 20
                    print(f"  >> speed: {per:.2f} s/step | projected {per * steps_per_epoch / 60:.1f} min/epoch "
                          f"({per * steps_per_epoch * epochs / 60:.0f} min for {epochs} epochs, plus evaluation)"
                          + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device.type == "cuda" else ""))
                if step % 100 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-300:]):.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
                if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                    print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping"); done = True; break
        vl = eval_loss(model, val_docs, val_pairs)
        qr = quick_rouge(model, val_docs, min(VAL_GEN_DOCS_EPOCH, len(val_docs)))
        rec = dict(epoch=ep, train_loss=ep_sum / max(ep_tok, 1), val_loss=vl, seconds=time.time() - t0, steps=step)
        if qr: rec.update(q_r1=qr["r1"]["f"], q_r2=qr["r2"]["f"], q_rL=qr["rL"]["f"])
        score = -vl if SELECT_BY == "loss" else (qr["r1"]["f"] + qr["rL"]["f"]) / 2
        if best["score"] is None or score > best["score"]:
            best = dict(score=score, epoch=ep, state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
            torch.save(best, RUN_DIR / "best_trainable.pt")
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step, tokens_seen=tok_seen, val_loss_init=v0,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    return history, best, stats

model = build_model()
history, best, stats = train_model(model, EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)
print(f"\nbest epoch {best['epoch']} (by {SELECT_BY}) | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

prepare tr:   0%|          | 0/6759 [00:00<?, ?it/s]

prepare val:   0%|          | 0/200 [00:00<?, ?it/s]

preparation 131s
TR: 6759 docs | chunks/doc mean 10.2 median 8 | chunk pairs with a target 47128 | empty chunks 31.9% | target tokens median 115 p90 231 | single-sentence cut 0.4%
VAL: 200 docs | chunks/doc mean 11.0 median 9 | chunk pairs with a target 1525 | empty chunks 30.6% | target tokens median 109 p90 232 | single-sentence cut 0.5%
adapters after ('o_proj', 'down_proj') in ('encoder', 'decoder'): 30 modules, bottleneck 64 -> 1,983,360 adapter params + 16,384 norm gains
identity-at-init check: max|adapted - Phase-1| = 0.00e+00
params: total 63.47M | trainable 2.000M (3.15%)
~58255 training sequences/epoch | ~1821 steps/epoch (32 seqs/step) | total 9105 steps
VAL loss before training (Phase-1 model, target tokens): 2.2399
  >> speed: 0.54 s/step | projected 16.4 min/epoch (82 min for 5 epochs, plus evaluation) | peak GPU 3.9 GB
  ep 1 step 100/9105 | loss 2.4505 | lr x0.366 | 55s
  ep 1 step 200/9105 | loss 1.7622 | lr x0.733 | 112s
  ep 1 step 300/9105 | loss 1.6095 | lr x1.000 

In [15]:
# Cell 21 — Smoke-test verdict + sample outputs (always printed)
val_extra = {}
_n = min(2, len(val_docs))
_s, _, _parts = summarize_docs(model, val_docs[:_n], return_parts=True)
for d, s in zip(val_docs[:_n], _s):
    print(f"--- {d['id']} | {len(d['chunks'])} chunks | generated {len(s.split())} words | reference {len(d['summ'].split())} words")
    print("GENERATED:", s[:700].replace("\n", " "))
    print("REFERENCE:", d["summ"][:700].replace("\n", " "))
if SMOKE_TEST:
    drop = stats["last_losses"] / stats["first_losses"]
    _ss = summarize_docs(model, val_docs)[0]
    m, _ = rouge_eval(_ss, [d["summ"] for d in val_docs])
    val_extra.update(smoke_r1=m["r1"]["f"], smoke_r1_p=m["r1"]["p"], smoke_r1_r=m["r1"]["r"], smoke_rL=m["rL"]["f"],
                     smoke_words_gen=float(np.mean([len(x.split()) for x in _ss])), smoke_words_ref=float(np.mean([len(d["summ"].split()) for d in val_docs])),
                     smoke_gen_stats=dict(LAST_GEN_STATS))
    print(f"generated {val_extra['smoke_words_gen']:.0f} words vs reference {val_extra['smoke_words_ref']:.0f} | R1 P {m['r1']['p']:.3f} R {m['r1']['r']:.3f} | "
          f"chunks with an empty output {LAST_GEN_STATS['empty_share']:.2f} | tokens/chunk {LAST_GEN_STATS['tokens_per_chunk']:.0f}")
    print(f"\nloss {stats['first_losses']:.3f} -> {stats['last_losses']:.3f} (x{drop:.2f}) | VAL loss {history[-1]['val_loss']:.3f} "
          f"(Phase-1 {stats['val_loss_init']:.3f}) | R1 {m['r1']['f']:.3f} RL {m['rL']['f']:.3f} on {len(val_docs)} docs")
    print("SMOKE VERDICT:", "OK -> run the full job" if (drop < 0.95 and history[-1]["val_loss"] < stats["val_loss_init"]) else
          "NOT OK: loss did not fall / VAL loss did not improve -> paste the log to Claude")
    print("Check the '>> speed' line: it projects minutes per epoch; set EPOCHS / TIME_BUDGET_MIN accordingly.")

--- 2459 | 12 chunks | generated 45 words | reference 912 words
GENERATED: The first respondent, who had contested the election, was declared elected to the Andhra Pradesh Legislative Assembly. The second respondent filed an Election Petition in the High Court challenging the election of the first respondent to the Andhra State Legislative Assembly from the Kollapur constituency.
REFERENCE: The election of the first respondent to the Andhra Pradesh Legislative Assembly at the General Election hold in 1967 was challenged by the appellants in an election petition. Various corrupt practices were alleged appellants to have been committed by the first respondent, his agents and supporters. One of these was that the first respondent paid a bribe to one R who had also filed nomination papers, in order to secure his withdrawal as a candidate. After the issues were framed the appellants made an application to the Court for impleading R but it was dismissed. The first respondent then filed an ap

## Decoding sweep and VAL evaluation (full runs)
The decoding configuration is chosen on the first `SWEEP_DOCS` VAL documents by (R1 + RL)/2, then **all 200** VAL documents are summarised with it (the audit's baselines were scored on the same 200). The analysis covers what the PDF asks for: scores vs the extractive baselines, length vs score, redundancy (repeated n-grams), abstractiveness of the outputs (novel n-grams vs the document), scores by document length, and cost.

In [16]:
# Cell 22 — Decoding sweep (on VAL[:SWEEP_DOCS]) -> best GEN
sweep_res = []
if not SMOKE_TEST and SWEEP_DOCS and DECODE_SWEEP:
    _docs = val_docs[:SWEEP_DOCS]; _refs = [d["summ"] for d in _docs]
    for cfg in DECODE_SWEEP:
        s, secs = summarize_docs(model, _docs, cfg); m, _ = rouge_eval(s, _refs)
        wg = float(np.mean([len(x.split()) for x in s]))
        sweep_res.append(dict(cfg=cfg, r1=m["r1"]["f"], r2=m["r2"]["f"], rL=m["rL"]["f"], r1_p=m["r1"]["p"], r1_r=m["r1"]["r"], words=wg,
                              empty_chunks=LAST_GEN_STATS["empty_share"], secs=secs))
        print(f"  {cfg}: R1 {m['r1']['f']:.4f} (P {m['r1']['p']:.3f} R {m['r1']['r']:.3f}) R2 {m['r2']['f']:.4f} RL {m['rL']['f']:.4f} | "
              f"words {wg:.0f} | empty chunks {LAST_GEN_STATS['empty_share']:.2f} | {secs:.0f}s")
    bestcfg = max(sweep_res, key=lambda r: (r["r1"] + r["rL"]) / 2)["cfg"]
    GEN.update(bestcfg); print("chosen decoding:", bestcfg)
else:
    print("decoding sweep skipped")

  {'no_repeat_ngram': 4, 'rep_penalty': 1.1, 'min_new_all': 0}: R1 0.2153 (P 0.777 R 0.147) R2 0.1201 RL 0.1478 | words 115 | empty chunks 0.84 | 22s
  {'no_repeat_ngram': 4, 'rep_penalty': 1.1, 'min_new_all': 32}: R1 0.5847 (P 0.701 R 0.543) R2 0.3350 RL 0.3133 | words 662 | empty chunks 0.00 | 40s
  {'no_repeat_ngram': 4, 'rep_penalty': 1.1, 'min_new_all': 64}: R1 0.6058 (P 0.643 R 0.623) R2 0.3433 RL 0.3121 | words 844 | empty chunks 0.00 | 40s
  {'no_repeat_ngram': 3, 'rep_penalty': 1.0, 'min_new_all': 32}: R1 0.5830 (P 0.686 R 0.550) R2 0.3295 RL 0.3104 | words 696 | empty chunks 0.00 | 40s
  {'no_repeat_ngram': 0, 'rep_penalty': 1.2, 'min_new_all': 32}: R1 0.5774 (P 0.706 R 0.528) R2 0.3257 RL 0.3070 | words 644 | empty chunks 0.00 | 40s
chosen decoding: {'no_repeat_ngram': 4, 'rep_penalty': 1.1, 'min_new_all': 64}


In [17]:
# Cell 23 — VAL evaluation: ROUGE (+ BERTScore), lengths, redundancy, abstractiveness, by document length
val_res = {}
if not SMOKE_TEST:
    refs = [d["summ"] for d in val_docs]
    val_sum, t_val = summarize_docs(model, val_docs)
    val_extra["gen_stats"] = dict(LAST_GEN_STATS)
    m, per = rouge_eval(val_sum, refs)
    val_res = {k: v["f"] for k, v in m.items()}
    val_extra.update(seconds=t_val, sec_per_doc=t_val / len(val_docs),
                     words_gen=float(np.mean([len(s.split()) for s in val_sum])), words_ref=float(np.mean([len(r.split()) for r in refs])),
                     r1_p=m["r1"]["p"], r1_r=m["r1"]["r"])
    if RUN_BERTSCORE and device.type == "cuda":
        try:
            import bert_score
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True); import bert_score
        import transformers; transformers.logging.set_verbosity_error()
        _bs = bert_score.BERTScorer(model_type=BERTSCORE_MODEL, lang="en", device="cuda", batch_size=8)
        P_, R_, F_ = _bs.score(val_sum, refs)
        val_res.update(bs_p=float(P_.mean()), bs_r=float(R_.mean()), bs_f=float(F_.mean())); per["bs"] = F_.numpy()
        del _bs; gc.collect(); torch.cuda.empty_cache()
    print(f"===== VAL ({len(val_docs)} docs) | R1 {val_res['r1']:.4f} (P {m['r1']['p']:.3f} R {m['r1']['r']:.3f}) | R2 {val_res['r2']:.4f} | "
          f"RL {val_res['rL']:.4f}" + (f" | BERTScore F1 {val_res['bs_f']:.4f}" if "bs_f" in val_res else ""))
    for name, b in BASELINES.items():
        print(f"      vs {name:6s}: R1 {val_res['r1'] - b['r1']:+.4f} | R2 {val_res['r2'] - b['r2']:+.4f} | RL {val_res['rL'] - b['rL']:+.4f}"
              + (f" | BERTScore {val_res['bs_f'] - b['bs']:+.4f}" if "bs_f" in val_res else ""))
    print(f"      words: generated {val_extra['words_gen']:.0f} vs reference {val_extra['words_ref']:.0f} | {val_extra['sec_per_doc']:.1f} s/doc")

    # redundancy: share of repeated 3-grams inside each output; abstractiveness: novel n-grams w.r.t. the document
    def rep3(t):
        w = words(t); g = list(zip(w, w[1:], w[2:]))
        return 1 - len(set(g)) / max(len(g), 1)
    def novel(t, d, n):
        a, s = words(d), words(t); ga = set(zip(*[a[i:] for i in range(n)])); gs = list(zip(*[s[i:] for i in range(n)]))
        return sum(x not in ga for x in gs) / max(len(gs), 1)
    val_extra["rep3_gen"] = float(np.mean([rep3(s) for s in val_sum])); val_extra["rep3_ref"] = float(np.mean([rep3(r) for r in refs]))
    val_extra["novel"] = {n: float(np.mean([novel(s, d["doc"], n) for s, d in zip(val_sum, val_docs)])) for n in (1, 2, 4)}
    print(f"      redundancy (repeated 3-grams): generated {val_extra['rep3_gen']:.3f} vs reference {val_extra['rep3_ref']:.3f}")
    print(f"      novel n-grams vs the document (1/2/4): {[round(v, 3) for v in val_extra['novel'].values()]}  (references: 0.044 / 0.223 / 0.484)")

    ntok_v = np.asarray([d["ntok"] for d in val_docs]); cuts = np.percentile(ntok_v, [33.3, 66.7]); bk = np.digitize(ntok_v, cuts); rows = []
    print("      by document length (token terciles):")
    for b_, name in enumerate(("short", "medium", "long")):
        sel = bk == b_
        r_ = dict(bucket=name, n=int(sel.sum()), median_tokens=float(np.median(ntok_v[sel])), r1=float(per["r1"][sel].mean()),
                  rL=float(per["rL"][sel].mean()), truncated=float(np.mean([len(val_docs[i]["chunks"]) > GEN["max_chunks"] for i in np.where(sel)[0]])))
        if "bs" in per: r_["bs"] = float(per["bs"][sel].mean())
        rows.append(r_)
        print(f"        {name:6s}: {sel.sum():3d} docs | median {r_['median_tokens']:6.0f} tokens | R1 {r_['r1']:.3f} RL {r_['rL']:.3f}"
              + (f" BS {r_['bs']:.3f}" if "bs" in r_ else "") + f" | chunks skipped in {100 * r_['truncated']:.0f}% of docs")
    val_extra["by_length"] = rows
    (RUN_DIR / "val_summaries.json").write_text(json.dumps([dict(id=d["id"], summary=s) for d, s in zip(val_docs, val_sum)], ensure_ascii=False))
else:
    print("no VAL evaluation (smoke run)")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

===== VAL (200 docs) | R1 0.5956 (P 0.621 R 0.632) | R2 0.3355 | RL 0.3092 | BERTScore F1 0.8555
      vs tfidf : R1 +0.0207 | R2 +0.0169 | RL +0.0084 | BERTScore +0.0022
      vs lead  : R1 +0.0734 | R2 +0.0847 | RL +0.0655 | BERTScore +0.0234
      vs oracle: R1 -0.1904 | R2 -0.2687 | RL -0.2249 | BERTScore -0.0300
      words: generated 805 vs reference 848 | 0.6 s/doc
      redundancy (repeated 3-grams): generated 0.115 vs reference 0.099
      novel n-grams vs the document (1/2/4): [0.033, 0.158, 0.379]  (references: 0.044 / 0.223 / 0.484)
      by document length (token terciles):
        short :  67 docs | median   2033 tokens | R1 0.591 RL 0.340 BS 0.863 | chunks skipped in 0% of docs
        medium:  66 docs | median   4260 tokens | R1 0.600 RL 0.312 BS 0.857 | chunks skipped in 0% of docs
        long  :  67 docs | median   8362 tokens | R1 0.596 RL 0.276 BS 0.847 | chunks skipped in 54% of docs


In [18]:
# Cell 24 — Test summaries -> prediction.json (list format) -> submission.zip (full runs only)
def verify_submission(zip_path, docs):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list), "prediction.json must be a JSON list of {id, summary}"
    want = [str(d["id"]) for d in docs]; got = [str(e["id"]) for e in sub]
    assert len(got) == len(set(got)), "duplicate ids"
    assert set(got) == set(want), f"id mismatch: missing {len(set(want) - set(got))}, extra {len(set(got) - set(want))}"
    for e in sub:
        assert set(e) == {"id", "summary"}, f"unexpected keys {set(e)}"
        assert isinstance(e["summary"], str) and e["summary"].strip(), f"{e['id']}: empty summary"
    print(f"verified: {len(sub)} summaries, one non-empty string per test id")

if not SMOKE_TEST:
    prepare_docs(TEST_ALL, "test", False)
    test_sum, t_test = summarize_docs(model, TEST_ALL)
    sub = [dict(id=d["id"], summary=s) for d, s in zip(TEST_ALL, test_sum)]          # id keeps its type from test.json
    out_dir = RUN_DIR / "submission"; out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")
    verify_submission(out_dir / "submission.zip", TEST_ALL)
    print(f"test: {len(sub)} docs in {t_test:.0f}s | words/summary mean {np.mean([len(s.split()) for s in test_sum]):.0f}")
else:
    print("smoke run: no submission written")

prepare test:   0%|          | 0/100 [00:00<?, ?it/s]

verified: 100 summaries, one non-empty string per test id
test: 100 docs in 60s | words/summary mean 781


In [19]:
# Cell 25 — Save trainable weights + run log
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, Path): return str(o)
    return o
log = dict(run=RUN_NAME, method=METHOD, hp=HP[METHOD], chunk_tokens=CHUNK_TOKENS, tgt_max=TGT_MAX, empty_keep=EMPTY_KEEP,
           min_align_f1=MIN_ALIGN_F1, epochs=EPOCHS, micro_tokens=MICRO_TOKENS, step_seqs=STEP_SEQS, select_by=SELECT_BY,
           gen=GEN, decode_sweep=sweep_res, params_total=model.n_total, params_trainable=model.n_trainable,
           history=history, best_epoch=best["epoch"], stats=stats, val=val_res, val_extra=val_extra, baselines=BASELINES,
           n_train_docs=len(train_docs), n_val_docs=len(val_docs), smoke=SMOKE_TEST, phase1_arch=lm_cfg.arch_name, enc_layers=lm_cfg.n_encoder_layers, dec_layers=lm_cfg.n_decoder_layers)
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "method": METHOD, "hp": HP[METHOD], "gen": GEN, "log": _jsonable(log)}, RUN_DIR / f"summ_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

saved: ['best_trainable.pt', 'run_log.json', 'submission', 'summ_p6_encdec_adapter.pt', 'val_summaries.json']


## KD teacher export (for run #3)
Two signals, both keyed by `"<doc uid>|<chunk index>"` so the student notebook (same tokenizer, same chunking and alignment code) can match them exactly:
* **Word-level KD** (Hinton et al. 2015; Kim & Rush 2016): for every TR chunk pair with a target, the teacher's **top-k logits** at each gold target position (+ the log-sum-exp over the full vocabulary, so the student can recover true probabilities of the top-k tokens). Possible across architectures (this encoder-decoder → a decoder-only student) only because all Phase-1 models share one tokenizer. Targets here are cut at 255 tokens (run #1: 384), so for the few long targets the student uses the teacher's logits on the common prefix only.
* **Sequence-level KD** (Kim & Rush 2016): the teacher's own greedy summaries for up to `TEACHER_GEN_PAIRS` TR chunks (with the chosen decoding config). The student trains on them as targets.

In [20]:
# Cell 26 — Export teacher signals
if EXPORT_TEACHER and not SMOKE_TEST:
    t0 = time.time(); model.eval()
    tr_pairs = [(di, j) for di, d in enumerate(train_docs) for j, t in enumerate(d["tgts"]) if t is not None]
    topk = {}
    with torch.no_grad():
        for b in tqdm(make_microbatches(train_docs, tr_pairs, shuffle=False, budget=2 * MICRO_TOKENS), desc="teacher top-k", leave=False):
            bt = build_batch(train_docs, b)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                h, y = forward_hidden(model, bt)
            for r, (di, j) in enumerate(b):
                sel = y[r] != -100
                lg = model.lm_head(h[r][sel].float()).float()
                v, ix = lg.topk(TEACHER_TOPK, dim=-1); lse = torch.logsumexp(lg, -1)
                topk[f"{train_docs[di]['uid']}|{j}"] = (v.half().cpu().numpy(), ix.to(torch.int32).cpu().numpy().astype(np.uint16),
                                                        lse.cpu().numpy().astype(np.float32))
    print(f"top-k logits for {len(topk)} pairs in {time.time() - t0:.0f}s")
    t1 = time.time(); gen_out = {}
    rng = random.Random(SEED); cand = list(tr_pairs); rng.shuffle(cand); cand = cand[:TEACHER_GEN_PAIRS]
    cand.sort(key=lambda p: len(prompt_ids(train_docs[p[0]], p[1])))
    for s in range(0, len(cand), GEN["batch"]):
        b = cand[s:s + GEN["batch"]]
        res = generate(model, [prompt_ids(train_docs[di], j) for di, j in b], GEN["max_new"], None, GEN["no_repeat_ngram"], GEN["rep_penalty"])
        for (di, j), o in zip(b, res): gen_out[f"{train_docs[di]['uid']}|{j}"] = np.asarray(o, dtype=np.uint16)
        if (time.time() - t1) / 60 > TEACHER_GEN_MIN:
            print(f"TEACHER_GEN_MIN reached after {len(gen_out)} chunks"); break
    print(f"teacher generations for {len(gen_out)} chunks in {time.time() - t1:.0f}s")
    meta = dict(teacher=RUN_NAME, arch="encdec", method=METHOD, hp=HP[METHOD], chunk_tokens=CHUNK_TOKENS, tgt_max=TGT_MAX, min_align_f1=MIN_ALIGN_F1,
                max_full_chars=MAX_FULL_CHARS, tag_template=TAG_TEMPLATE, sep_text=SEP_TEXT, topk=TEACHER_TOPK, gen=GEN,
                params_total=int(model.n_total), params_trainable=int(model.n_trainable), val=val_res, val_checksum=VAL_CHECKSUM_EXPECTED)
    path = RUN_DIR / f"teacher_{RUN_NAME}.pt"
    torch.save(dict(meta=meta, topk=topk, gen=gen_out), path)
    print(f"saved {path} ({path.stat().st_size / 1e6:.0f} MB) | total {time.time() - t0:.0f}s")
else:
    print("teacher export skipped")

teacher top-k:   0%|          | 0/1247 [00:00<?, ?it/s]

top-k logits for 47128 pairs in 355s
teacher generations for 15000 chunks in 445s
saved /kaggle/working/runs/p6_encdec_adapter/teacher_p6_encdec_adapter.pt (283 MB) | total 804s


In [21]:
# Cell 27 — PASTE TO CLAUDE
short = dict(run=RUN_NAME, smoke=SMOKE_TEST, hp=HP[METHOD], params_trainable=model.n_trainable, params_total=model.n_total,
             history=history, best_epoch=best["epoch"], stats={k: (round(v, 4) if isinstance(v, float) else v) for k, v in stats.items()},
             gen=GEN, decode_sweep=sweep_res, val={k: round(v, 4) for k, v in val_res.items()},
             val_extra={k: v for k, v in val_extra.items() if k != "seconds"})
print("================ PASTE TO CLAUDE ================")
print(json.dumps(_jsonable(short), separators=(",", ":")))
print("=================================================")

================ PASTE TO CLAUDE ================
{"run":"p6_encdec_adapter","smoke":false,"hp":{"lr":0.001,"wd":0.0,"bottleneck":64,"sites":["attn","ffn"],"stacks":["encoder","decoder"],"train_norms":true},"params_trainable":1999744,"params_total":63473536,"history":[{"epoch":1,"train_loss":1.634346059494136,"val_loss":1.5688008718577522,"seconds":852.0198752880096,"steps":1477,"q_r1":0.15309555801086538,"q_r2":0.08476461782523811,"q_rL":0.11068142442493423},{"epoch":2,"train_loss":1.5209590861240885,"val_loss":1.5257674891590818,"seconds":851.5960853099823,"steps":2954,"q_r1":0.1692900693339773,"q_r2":0.09278770929799325,"q_rL":0.12638396507035896},{"epoch":3,"train_loss":1.470803127036914,"val_loss":1.5059576925550715,"seconds":851.8241505622864,"steps":4426,"q_r1":0.17699123307331113,"q_r2":0.09846535423961303,"q_rL":0.130055527875084},{"epoch":4,"train_loss":1.430428221786553,"val_loss":1.5169764969520723,"seconds":854.4723787307739,"steps":5900,"q_r1":0.16359531492874724,"q_r2":0